# Aggregations and CTEs

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-02-aggregations-and-ctes.ipynb)

The basics chapter introduced `COUNT`, `SUM` and `GROUP BY`. This chapter goes further. It covers the aggregates
you will use most, grouping at several levels, subqueries that feed a query, and **common table expressions**
(CTEs), which name each step of a long query so it reads from top to bottom.

It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

## 1. More aggregate functions

`COUNT(DISTINCT column)` counts the different values in a column. An aggregate can take an expression, not only a
column name, so `SUM(amount * 0.1)` sums ten per cent of each order.

`MIN` and `MAX` also work on dates, which makes them the usual tools for "first seen" and "last seen".

### Where you'll use it

- **Unique users:** `COUNT(DISTINCT user_id)` for monthly active users, the most common distinct count in product analytics.
- **Percentiles:** median order value or p95 latency for service-level reporting.

In [ ]:
spark.sql("""
SELECT
  COUNT(DISTINCT customer_id) AS buyers,
  COUNT(DISTINCT status)      AS statuses,
  MIN(order_date)             AS first_order,
  MAX(order_date)             AS last_order,
  ROUND(SUM(amount * 0.1), 2) AS tenth_of_all_amounts
FROM orders
""").show()

## 2. Grouping at several levels

Listing more than one column after `GROUP BY` creates one group per combination, which is useful for drill-downs.

`ROLLUP` adds subtotals. `GROUP BY ROLLUP(a, b)` returns the detail rows for each `(a, b)` pair, one subtotal row
per `a`, and a grand total. In the subtotal rows, the grouped column that was rolled up is `NULL`.

### Where you'll use it

- **Financial reports:** totals by region, then by region and product, with a grand total on the last line.
- **Dashboards with drill-down:** one query returns the subtotals and the detail together.

In [ ]:
spark.sql("""
SELECT c.region, o.status, COUNT(*) AS orders, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.region, o.status
ORDER BY c.region, o.status
""").show()

spark.sql("""
SELECT c.region, o.status, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY ROLLUP(c.region, o.status)
ORDER BY c.region, o.status
""").show()

### Telling subtotals apart: GROUPING()

In a `ROLLUP` result, a NULL can mean a real NULL in the data, or a subtotal row. `GROUPING(column)` returns 1 for a
subtotal row and 0 for a detail row, so you can label the rows explicitly. `CUBE(a, b)` goes further and produces a
subtotal for every combination of the columns, not only the left-to-right prefixes that `ROLLUP` makes.

In [ ]:
spark.sql("""
SELECT region, status, total
FROM (
  SELECT CASE WHEN GROUPING(c.region) = 1 THEN 'all regions' ELSE c.region END AS region,
         CASE WHEN GROUPING(o.status) = 1 THEN 'all statuses' ELSE o.status END AS status,
         SUM(o.amount) AS total,
         GROUPING(c.region) AS region_total,
         GROUPING(o.status) AS status_total
  FROM orders o
  JOIN customers c ON c.customer_id = o.customer_id
  GROUP BY ROLLUP(c.region, o.status)
) t
ORDER BY region_total, region, status_total, status
""").show(truncate=False)

## 3. Subqueries

A **subquery** is a query inside another query. Where it appears decides what it returns.

- A **scalar subquery** returns one value, and can stand wherever a value can.
- An **IN** subquery returns a list of values to test against.
- An **EXISTS** subquery returns true or false for each row.
- A **correlated** subquery refers to the outer row, so it runs once per row.

### Where you'll use it

- **Comparing a row to its group:** orders larger than the customer's own average.
- **Existence checks:** customers with at least one refund, using `EXISTS`.

In [ ]:
spark.sql("""
SELECT c.name,
       (SELECT COUNT(*) FROM orders o WHERE o.customer_id = c.customer_id) AS orders
FROM customers c
ORDER BY c.customer_id
""").show()

spark.sql("""
SELECT name FROM customers
WHERE customer_id IN (SELECT customer_id FROM orders WHERE status = 'cancelled')
ORDER BY name
""").show()

spark.sql("""
SELECT order_id, customer_id, amount
FROM orders o
WHERE amount > (SELECT AVG(x.amount) FROM orders x WHERE x.customer_id = o.customer_id)
ORDER BY order_id
""").show()

The last query is correlated: for each order, the subquery computes the average of **that customer's** orders. It
returns the orders larger than their own customer's average.

A subquery in the `FROM` clause is also common. It gives a name to an intermediate result:

In [ ]:
spark.sql("""
SELECT customer_id, total
FROM (SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id) t
WHERE total >= 500
ORDER BY customer_id
""").show()

## 4. Common table expressions (CTEs)

A CTE is a named query placed before the main query with `WITH`:

```sql
WITH step1 AS (...),
     step2 AS (...)
SELECT ... FROM step2;
```

Each step can use the steps before it. The main query reads the last step. Long queries become a sequence of named
steps, and each step can be checked on its own. A CTE is visible only inside the query that defines it.

The query below answers "which regions had more than 1,000 in completed orders?" in three named steps:

### Where you'll use it

- **Readable transformations:** a dbt model is a chain of CTEs, one per logical step, that a reviewer can follow.
- **Reusing an intermediate result:** a CTE of completed orders used in two later steps.

In [ ]:
spark.sql("""
WITH completed AS (
  SELECT customer_id, amount
  FROM orders
  WHERE status = 'completed'
),
by_customer AS (
  SELECT customer_id, SUM(amount) AS total
  FROM completed
  GROUP BY customer_id
),
by_region AS (
  SELECT c.region, SUM(b.total) AS total
  FROM by_customer b
  JOIN customers c ON c.customer_id = b.customer_id
  GROUP BY c.region
)
SELECT region, total
FROM by_region
WHERE total > 1000
ORDER BY region
""").show()

### Recursive CTEs

A CTE can refer to itself. That is how SQL walks a hierarchy, such as an employee's chain of managers. A recursive
CTE has two parts joined by `UNION ALL`: a starting row, then a step that adds the rows one level further down. The
`staff` table below has a `manager_id` for each person.

Spark 4.0, the version Colab installs today, fails to plan a recursive CTE with an internal error. The query is
therefore shown as a reference and is not run here. The example below walks the same hierarchy with a self-join,
which works on every version and covers a fixed number of levels.

### Where you'll use it

- **Org charts and bills of materials:** walking a manager chain or a parts hierarchy.
- **Graph traversal:** finding all accounts linked to a flagged account through several hops.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW staff AS
SELECT * FROM VALUES
  (1, 'Alice', NULL),
  (2, 'Bob',   1),
  (3, 'Carol', 1),
  (4, 'Dave',  2),
  (5, 'Eve',   2)
AS staff(emp_id, name, manager_id)
""")

Reference only, not executed. Spark 4.1 and later run this form of recursive CTE:

```sql
WITH RECURSIVE reports(emp_id, name, level) AS (
  SELECT emp_id, name, 0 FROM staff WHERE name = 'Alice'
  UNION ALL
  SELECT s.emp_id, s.name, r.level + 1
  FROM staff s
  JOIN reports r ON s.manager_id = r.emp_id
)
SELECT name, level FROM reports WHERE level > 0 ORDER BY emp_id
```

In [ ]:
spark.sql("""
SELECT s.emp_id, s.name, 1 AS level
FROM staff s
JOIN staff boss ON s.manager_id = boss.emp_id
WHERE boss.name = 'Alice'
UNION ALL
SELECT s.emp_id, s.name, 2
FROM staff s
JOIN staff mid ON s.manager_id = mid.emp_id
JOIN staff boss ON mid.manager_id = boss.emp_id
WHERE boss.name = 'Alice'
ORDER BY emp_id
""").show()

## 5. Choosing between a join, a subquery and a CTE

| Use | When |
| --- | --- |
| A **join** | You need columns from both tables in the result. |
| A **subquery** or **EXISTS** | You only need to test for a match, or you need one value per row. |
| A **CTE** | The logic has several named steps, or you use the same intermediate result more than once. |

A CTE does not change the answer, only how readable the query is. When a subquery and a CTE give the same result,
prefer the one a reader can follow more easily.

### Where you'll use it

- **Code review:** the team's SQL style guide usually asks for CTEs for multi-step logic, because a reviewer can read them top to bottom.

## Interview notes

- **Group before you join when you can.** Aggregating a large table first and joining the small result is usually
  cheaper than joining everything and then aggregating.
- **COUNT(DISTINCT) is expensive.** It must track every distinct value. For very large data, engines offer
  approximate versions, such as `approx_count_distinct`.
- **A correlated subquery runs per row** in the logical model. Engines often rewrite it into a join, but a
  correlated subquery you cannot rewrite can be slow; prefer a join or a CTE that computes the value once per group.

## Summary

- Use `COUNT(DISTINCT ...)`, expressions inside aggregates, and `ROLLUP` for subtotals.
- A correlated subquery refers to the outer row. An `EXISTS` subquery tests for a match.
- A CTE names each step. A recursive CTE walks a hierarchy on versions that support it; a self-join covers a fixed depth.

## Practice questions

1. How many customers have a completed order, and when was the latest completed order placed? Show both in one row.
2. Which orders are larger than their own customer's average order amount, counting every status? Show `order_id`, `customer_id` and `amount`, sorted by `order_id`.
3. Which customers have at least one cancelled order? Use `EXISTS`, and show their names sorted by name.
4. Using a CTE, show each region's total of completed orders, sorted by region.
5. Using a self-join, list everyone who reports to Alice, directly or through one other person, with their level below Alice. Sort by `emp_id`.

## Answers

### Question 1

In [ ]:
q = """
SELECT COUNT(DISTINCT customer_id) AS customers, MAX(order_date) AS latest_order
FROM orders
WHERE status = 'completed'
"""
spark.sql(q).show()
from datetime import date
assert spark.sql(q).collect() == [(6, date(2023, 3, 30))]

### Question 2

In [ ]:
q = """
SELECT order_id, customer_id, amount
FROM orders o
WHERE amount > (SELECT AVG(x.amount) FROM orders x WHERE x.customer_id = o.customer_id)
ORDER BY order_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 1, 200), (6, 2, 400), (8, 3, 180), (10, 4, 320), (11, 5, 500), (14, 6, 325)]

### Question 3

In [ ]:
q = """
SELECT c.name
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status = 'cancelled')
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Beta LLC",), ("Foxtrot GmbH",)]

### Question 4

In [ ]:
q = """
WITH completed AS (
  SELECT customer_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY customer_id
)
SELECT c.region, SUM(t.total) AS completed_total
FROM completed t
JOIN customers c ON c.customer_id = t.customer_id
GROUP BY c.region
ORDER BY c.region
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("North", 1150), ("South", 620), ("West", 1225)]

### Question 5

In [ ]:
q = """
SELECT s.emp_id, s.name, 1 AS level
FROM staff s
JOIN staff boss ON s.manager_id = boss.emp_id
WHERE boss.name = 'Alice'
UNION ALL
SELECT s.emp_id, s.name, 2
FROM staff s
JOIN staff mid ON s.manager_id = mid.emp_id
JOIN staff boss ON mid.manager_id = boss.emp_id
WHERE boss.name = 'Alice'
ORDER BY emp_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, "Bob", 1), (3, "Carol", 1), (4, "Dave", 2), (5, "Eve", 2)]